In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression


In [4]:
df = pd.read_csv("/home/durga/Desktop/DURGA/E2S1/DS/Course/NIFTY-Stock-Dashboard/dataset/NIFTY50_all.csv")
df["Date"] = pd.to_datetime(df["Date"])

In [5]:
companies = df["Symbol"].unique()
companies
print(len(companies))
companies

65


array(['MUNDRAPORT', 'ADANIPORTS', 'ASIANPAINT', 'UTIBANK', 'AXISBANK',
       'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJAUTOFIN', 'BAJFINANCE', 'BHARTI',
       'BHARTIARTL', 'BPCL', 'BRITANNIA', 'CIPLA', 'COALINDIA', 'DRREDDY',
       'EICHERMOT', 'GAIL', 'GRASIM', 'HCLTECH', 'HDFC', 'HDFCBANK',
       'HEROHONDA', 'HEROMOTOCO', 'HINDALC0', 'HINDALCO', 'HINDLEVER',
       'HINDUNILVR', 'ICICIBANK', 'INDUSINDBK', 'INFOSYSTCH', 'INFY',
       'IOC', 'ITC', 'JSWSTL', 'JSWSTEEL', 'KOTAKMAH', 'KOTAKBANK', 'LT',
       'M&M', 'MARUTI', 'NESTLEIND', 'NTPC', 'ONGC', 'POWERGRID',
       'RELIANCE', 'SBIN', 'SHREECEM', 'SUNPHARMA', 'TELCO', 'TATAMOTORS',
       'TISCO', 'TATASTEEL', 'TCS', 'TECHM', 'TITAN', 'ULTRACEMCO',
       'UNIPHOS', 'UPL', 'SESAGOA', 'SSLT', 'VEDL', 'WIPRO', 'ZEETELE',
       'ZEEL'], dtype=object)

In [6]:
import joblib
results = []
for company in companies:
    print(f"Traing {company}")

    company_df=df[df["Symbol"]==company].copy()

    company_df=company_df.sort_values("Date")

    imputer=SimpleImputer(strategy="mean")
    cols=[
        "Trades",
        "Deliverable Volume",
        "%Deliverble"
    ]
    for col in cols:
        if company_df[col].isnull().all():
            print(f"{company}: {col} is completely missing. Filling with 0.")
            company_df[col] = 0
        else:
            company_df[col] = company_df[col].fillna(company_df[col].mean())
    company_df["Target"]=company_df["Close"].shift(-1)
    company_df=company_df.dropna()

    company_df["Price_Change"] = (
        company_df["Close"] - company_df["Open"]
    )

    company_df["High_Low_Difference"] = (
        company_df["High"] - company_df["Low"]
    )

    company_df["Daily_Return"] = (
        (company_df["Close"] - company_df["Open"])
        / company_df["Open"]
    ) * 100

    X = company_df[[
            "Prev Close",
            "Open",
            "High",
            "Low",
            "VWAP",
            "Volume",
            "Trades",
            "Deliverable Volume",
            "%Deliverble",
            "Price_Change",
            "High_Low_Difference",
            "Daily_Return"
        ]
    ]

    y = company_df["Target"]
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

    #scaler = StandardScaler()

    #X_train = scaler.fit_transform(X_train)
    #X_test = scaler.transform(X_test)

    model = LinearRegression()

    model.fit(X_train, y_train)

    joblib.dump(model, f"../models/{company}.pkl")

    

    print(f"{company} Model Saved")
    from sklearn.metrics import r2_score

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    results.append({
    "Company": company,
    "R2 Score": r2
    })



Traing MUNDRAPORT
MUNDRAPORT Model Saved
Traing ADANIPORTS
ADANIPORTS Model Saved
Traing ASIANPAINT
ASIANPAINT Model Saved
Traing UTIBANK
UTIBANK: Trades is completely missing. Filling with 0.
UTIBANK Model Saved
Traing AXISBANK
AXISBANK Model Saved
Traing BAJAJ-AUTO
BAJAJ-AUTO Model Saved
Traing BAJAJFINSV
BAJAJFINSV Model Saved
Traing BAJAUTOFIN
BAJAUTOFIN: Trades is completely missing. Filling with 0.
BAJAUTOFIN Model Saved
Traing BAJFINANCE
BAJFINANCE Model Saved
Traing BHARTI
BHARTI: Trades is completely missing. Filling with 0.
BHARTI Model Saved
Traing BHARTIARTL
BHARTIARTL Model Saved
Traing BPCL
BPCL Model Saved
Traing BRITANNIA
BRITANNIA Model Saved
Traing CIPLA
CIPLA Model Saved
Traing COALINDIA
COALINDIA Model Saved
Traing DRREDDY
DRREDDY Model Saved
Traing EICHERMOT
EICHERMOT Model Saved
Traing GAIL
GAIL Model Saved
Traing GRASIM
GRASIM Model Saved
Traing HCLTECH
HCLTECH Model Saved
Traing HDFC
HDFC Model Saved
Traing HDFCBANK
HDFCBANK Model Saved
Traing HEROHONDA
HEROHOND

In [7]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values("R2 Score", ascending=False)

print(results_df)

       Company  R2 Score
16   EICHERMOT  0.999508
47    SHREECEM  0.999434
2   ASIANPAINT  0.999423
12   BRITANNIA  0.999373
29  INDUSINDBK  0.999328
..         ...       ...
54       TECHM  0.970403
57     UNIPHOS  0.969990
0   MUNDRAPORT  0.963381
8   BAJFINANCE  0.942282
34      JSWSTL  0.464096

[65 rows x 2 columns]


In [8]:
import sklearn

print(sklearn.__version__)

1.9.0


In [9]:
results_df.to_csv("../model_results.csv", index=False)